## Tools in an LLM call

A model can only produce text. It can't check the weather, query a database or call an API. **Tool calling** lets the model **ask** your code to run a function, and then use the result in its answer.

The key idea: **the model never runs the tool itself.** It only returns the tool's name and arguments. Your code runs the tool and sends the result back.

### How it works

1. You give the model a list of tools (name, description, arguments).
2. The model decides whether it needs a tool and replies with a **tool call** instead of text.
3. Your code runs the tool.
4. You send the tool result back to the model.
5. The model writes the final answer.

### Step 1: Define a tool

```python
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"It is sunny and 25°C in {city}."   # fake data for demo
```

- `@tool` turns a normal function into a tool.
- The **docstring** is the description the model reads to decide when to use the tool. Write it clearly.
- The **type hints** (`city: str`) tell the model what arguments to send.

### Step 2: Attach tools to the model

```python
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:llama-3.3-70b-versatile")
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What is the weather in Paris?")

print(response.content)      # usually empty: the model chose a tool instead of answering
print(response.tool_calls)   # [{'name': 'get_weather', 'args': {'city': 'Paris'}, 'id': '...', ...}]
```

- `bind_tools([...])` returns a new model that knows about your tools.
- `response.tool_calls` is a list. Each item has the tool `name`, its `args` and a call `id`.
- If the question doesn't need a tool (for example "Hi!"), `tool_calls` is empty and the model answers normally.

### Step 3: Run the tool and send the result back

```python
messages = [{"role": "user", "content": "What is the weather in Paris?"}]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)                      # keep the model's tool call in the history

for tool_call in ai_msg.tool_calls:
    tool_msg = get_weather.invoke(tool_call) # runs the tool, returns a ToolMessage
    messages.append(tool_msg)

final = model_with_tools.invoke(messages)    # the model now sees the tool result
print(final.text)
```

- `get_weather.invoke(tool_call)` runs your function and wraps the result in a `ToolMessage` that carries the matching call `id`.
- The full history must be sent back: **user message → model's tool call → tool result**.
- `final.text` is the natural-language answer, for example "It's sunny and 25°C in Paris."

### Multiple tools

```python
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

model_with_tools = model.bind_tools([get_weather, add])
```

The model picks the right tool from the descriptions. It can also return **several tool calls** in one response, so always loop over `response.tool_calls`.

### Forcing a tool

By default, the model decides whether to use a tool. You can force it:

```python
model.bind_tools([get_weather], tool_choice="get_weather")
```

Some providers also accept `tool_choice="any"` to force the model to pick one of the tools.

### Tools vs agents

Steps 2 and 3 are one round of a loop. An **agent** repeats it for you: call the model, run the tools, send the results back, and repeat until the model gives a final answer.

### Key takeaways

1. A tool is a Python function the model can **request**. Your code runs it.
2. Use `@tool`, with a clear **docstring** and **type hints**.
3. `model.bind_tools([...])` attaches tools. Read the request from `response.tool_calls`.
4. Send the `ToolMessage` back with the full history so the model can write the final answer.
5. Not every model supports tool calling, so check your provider's model list.


In [1]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [2]:
## Groq models

from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Tell me a joke")
response.content


'Why did the scarecrow win an award?\n\nBecause he was outstanding in his field! 🌾'

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str) ->str:
    """Get weather for a location"""
    return f"It is sunny and 25°C in {location}."   # fake data for demo

model_with_tools=model.bind_tools([get_weather])



In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': '8q8t96djm', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 277, 'total_tokens': 303, 'completion_time': 0.067964267, 'completion_tokens_details': None, 'prompt_time': 0.018861551, 'prompt_tokens_details': None, 'queue_time': 0.050901948, 'total_time': 0.086825818}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0bf5a-89ab-7e01-a9fa-15ef457cf7e0-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '8q8t96djm', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 277, 'output_tokens': 26, 'total_tokens': 303}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [8]:
## Step 1: <odel generates tool calls
messages = [{"role":"user", "content":"What's weather in boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to the model for final response
final_response =model_with_tools.invoke(messages)
print(final_response.text)

It's currently sunny in Boston with a temperature of 25°C (77°F). Enjoy the nice weather! ☀️


In [9]:
messages

[{'role': 'user', 'content': "What's weather in boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '89rtzw61r', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 276, 'total_tokens': 302, 'completion_time': 0.067848255, 'completion_tokens_details': None, 'prompt_time': 0.018731081, 'prompt_tokens_details': None, 'queue_time': 0.055857598, 'total_time': 0.086579336}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0bf68-4a33-76b2-9e6c-f29a616db03e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '89rtzw61r', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 26, 'total_tokens': 302}),
 ToolMessage(content='It is sunny and